# Store Sales Forecasting - End-to-End Pipeline

This notebook executes data cleaning, feature engineering, saves processed datasets to `data/processed/`, and displays all 9 exploratory graphs from `main.ipynb`.

In [1]:
import sys
import os
sys.path.append("..")

from src.preprocessing.load_data import load_raw_data
from src.preprocessing.holidays import process_holidays
from src.preprocessing.oil import process_oil
from src.preprocessing.transactions import process_transactions
from src.preprocessing.clean_data import filter_pre_opening_days
from src.features.build_features import build_all_features
from src.visualization.plots import (
    plot_holiday_vs_sales,
    plot_sales_vs_oil,
    plot_periodogram,
    plot_weekly_seasonality,
    plot_monthly_seasonality,
    plot_autocorrelation,
    plot_pacf,
    plot_store_hierarchies,
    plot_promo_impact_all_families
)

## 1. Data Cleaning & Preprocessing

In [2]:
# Load raw datasets
df, stores_df, transactions_df, oil_df, holidays_df, test_df = load_raw_data("../data/raw")
print("Raw merged df shape:", df.shape)

# 1. Process Holidays
df = process_holidays(df, holidays_df)
test_df = process_holidays(test_df, holidays_df)

# 2. Process Oil Prices
df, full_oil_df = process_oil(df, oil_df)
test_df, _ = process_oil(test_df, oil_df)

# 3. Process Transactions (Linear regression imputation & closed days)
df, daily_sales = process_transactions(df, transactions_df)

# 4. Filter Pre-Opening Zero-Sales Days
df = filter_pre_opening_days(df)
print("Cleaned df shape:", df.shape)

Raw merged df shape: (3000888, 10)
Cleaned df shape: (2778831, 16)


## 2. Feature Engineering & Saving Processed Output

In [3]:
# Build all manufactured features
df = build_all_features(df)
print("Final Feature Set Shape:", df.shape)

# Save processed datasets to data/processed/
os.makedirs("../data/processed", exist_ok=True)
processed_path = "../data/processed/train_processed.parquet"
df.to_parquet(processed_path, index=False)
print(f"Processed training dataset saved to: {processed_path}")

Final Feature Set Shape: (2778831, 42)
Processed training dataset saved to: ../data/processed/train_processed.parquet


## 3. Exploratory Data Visualizations

In [4]:
# 1. Average Store Sales per Day: Holiday vs Non-Holiday
plot_holiday_vs_sales(df)

In [4]:
# 2. Relationship: Sales vs. Oil Price (30-Day Centered Moving Averages)
plot_sales_vs_oil(df, full_oil_df)

In [5]:
# 3. Periodogram of Total Daily Sales (Identifying Dominant Frequencies)
plot_periodogram(df)

In [6]:
# 4. Weekly Seasonal Plot: Sales Movement Over 7 Days Across All Weeks
plot_weekly_seasonality(df)

In [7]:
# 5. Monthly Seasonal Plot: Average Daily Sales by Month (2013 - 2017)
plot_monthly_seasonality(df)

In [8]:
# 6. Autocorrelation of Sales by Lag (Lags 1 to 31)
plot_autocorrelation(df)

In [10]:
# 7. Partial Autocorrelation Function (PACF) - Lags 1 to 31
plot_pacf(df)

In [11]:
# 8. 4-Panel Store Hierarchy Diagnostic Grid Plot
plot_store_hierarchies(df)

In [12]:
# 9. Promotion Impact Across All 33 Product Families
plot_promo_impact_all_families(df)